# CoDA-GQA-L x Qwen3-4B: Train & Publish

Fine-tune Qwen3-4B with CoDA-GQA-L differential attention on Colab Pro+ (A100 40GB).

**Pipeline:**
1. Install deps & clone repo
2. Load Qwen3-4B, swap attention layers to CoDA
3. Phase 1: Unbounded training (teach differential attention)
4. Phase 2: Bounded training (teach memory banks)
5. Evaluate PPL (unbounded + bounded)
6. Push to Hugging Face Hub

**Requirements:** Colab Pro+ with A100 40GB GPU.

---

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets huggingface_hub accelerate

# Clone and install CoDA-GQA-L
import os
if not os.path.exists("/content/CoDA-QGA-L"):
    !git clone https://github.com/anthony-maio/CoDA-GQA-L.git /content/CoDA-QGA-L
else:
    !cd /content/CoDA-QGA-L && git pull

!pip install -q -e /content/CoDA-QGA-L

In [ ]:
import copy
import json
import math
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from coda_gqa_l import (
    CoDAGQALandmarkPerf2,
    CoDAGQALandmarkStatePerf2,
    LlamaCoDAAdapter,
    Qwen3CoDAAdapter,
)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
print(f"Dtype: {dtype}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen3-4B"
HF_REPO_ID = "YOUR_USERNAME/Qwen3-4B-CoDA-GQA-L"  # <-- CHANGE THIS

# Training hyperparameters
SEQ_LEN = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8          # effective batch = 8 seqs * 2048 = 16K tokens/update
LR = 5e-5               # projection LR
LR_CODA = 1e-3           # CoDA param LR (theta, lambda, head_norm)
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Phase 1: Unbounded training
PHASE1_STEPS = 2000
FREEZE_MODE = "attention"  # train all attention params (Q/K/V/O + CoDA)

# Phase 2: Bounded training
PHASE2_STEPS = 1000
BOUNDED_WINDOW = 256
BOUNDED_ME = 64
BOUNDED_MS = 64
BOUNDED_BLOCK_SIZE = 256
PHASE2_LR_SCALE = 0.1

# Eval
EVAL_EVERY = 200
EVAL_TOKENS = 50_000
EVAL_BOUNDED_TOKENS = 20_000

# Output
OUTPUT_DIR = Path("/content/runs/qwen3-4b-coda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Phase 1: {PHASE1_STEPS} steps (unbounded)")
print(f"Phase 2: {PHASE2_STEPS} steps (bounded W={BOUNDED_WINDOW}, Me={BOUNDED_ME}, Ms={BOUNDED_MS})")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM} seqs x {SEQ_LEN} = {BATCH_SIZE * GRAD_ACCUM * SEQ_LEN:,} tok/update")

## 1. Load Model & Swap Attention

In [ ]:
# Load Qwen3-4B
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=dtype,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Params: {total_params:,}")
print(f"Config: D={model.config.hidden_size}, H={model.config.num_attention_heads}, "
      f"Hkv={model.config.num_key_value_heads}, layers={model.config.num_hidden_layers}")
print(f"rope_theta={model.config.rope_theta:,}")

if device.type == "cuda":
    print(f"VRAM after model load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Swap all attention layers to CoDA-GQA (unbounded for Phase 1)
# Qwen3 uses the same attention layout as Llama: separate Q/K/V/O projections
# The adapter handles weight transfer, RoPE convention, and rope_theta automatically.

adapters: List[Qwen3CoDAAdapter] = []

for i, block in enumerate(model.model.layers):
    adapter = Qwen3CoDAAdapter.from_qwen3_attention(
        block.self_attn,
        bounded=False,
        head_norm_mode="identity",   # preserve pre-trained output scale
        rope_interleaved=False,       # Qwen3 uses contiguous-half RoPE
        theta_init=0.0,              # near-identity init (noise = signal initially)
        lambda_init_bias=-6.0,       # lambda ~0.0025 (small, gradients flow)
    )
    adapter = adapter.to(device=device, dtype=dtype)
    block.self_attn = adapter
    adapters.append(adapter)

print(f"Swapped {len(adapters)}/{len(model.model.layers)} attention layers")
print(f"  Mode: unbounded (Phase 1)")
print(f"  head_norm_mode=identity, theta_init=0.0")

if device.type == "cuda":
    print(f"VRAM after swap: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Enable gradient checkpointing (saves ~40% VRAM)
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    model.config.use_cache = False
    print("Gradient checkpointing: enabled")

In [ ]:
# Freeze non-attention parameters, configure optimizer param groups

# Freeze everything first
for p in model.parameters():
    p.requires_grad = False

# Unfreeze based on freeze mode
coda_param_names = {"lambda_proj", "theta", "head_norm"}

if FREEZE_MODE == "coda-only":
    # Only train CoDA-specific params (~50K params)
    for adapter in adapters:
        for name, p in adapter.named_parameters():
            if any(cn in name for cn in coda_param_names):
                p.requires_grad = True
elif FREEZE_MODE == "attention":
    # Train all attention params (Q/K/V/O + CoDA)
    for adapter in adapters:
        for p in adapter.parameters():
            p.requires_grad = True
elif FREEZE_MODE == "all":
    for p in model.parameters():
        p.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {n_train:,} / {n_total:,} ({100 * n_train / n_total:.1f}%)")

# Build optimizer with separate LR for CoDA params vs projections
adapter_param_ids = set()
for adapter in adapters:
    for p in adapter.parameters():
        adapter_param_ids.add(id(p))

coda_params, proj_params, other_params = [], [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if id(p) in adapter_param_ids:
        if any(cn in name for cn in coda_param_names):
            coda_params.append(p)
        else:
            proj_params.append(p)
    else:
        other_params.append(p)

groups = []
if coda_params:
    groups.append({"params": coda_params, "lr": LR_CODA, "weight_decay": 0.0})
    print(f"  CoDA params: {sum(p.numel() for p in coda_params):,} @ lr={LR_CODA}")
if proj_params:
    groups.append({"params": proj_params, "lr": LR, "weight_decay": WEIGHT_DECAY})
    print(f"  Proj params: {sum(p.numel() for p in proj_params):,} @ lr={LR}")
if other_params:
    groups.append({"params": other_params, "lr": LR, "weight_decay": WEIGHT_DECAY})
    print(f"  Other params: {sum(p.numel() for p in other_params):,} @ lr={LR}")

optimizer = torch.optim.AdamW(groups)

## 2. Prepare Data

In [ ]:
# Load and tokenize WikiText-103 (training) and WikiText-2 (eval)

def load_and_chunk(tokenizer, dataset_name, split, seq_len, max_tokens=None):
    """Load dataset, tokenize, chunk into fixed-length sequences."""
    cache_dir = Path("/content/.token_cache")
    cache_dir.mkdir(exist_ok=True)
    import hashlib
    key = f"{MODEL_NAME}|{dataset_name}|{seq_len}|{split}"
    h = hashlib.md5(key.encode()).hexdigest()[:12]
    cache_path = cache_dir / f"{h}.pt"

    if cache_path.exists():
        print(f"  Loading cached tokens from {cache_path}")
        chunks = torch.load(cache_path, map_location="cpu", weights_only=True)
        print(f"  {chunks.shape[0]:,} chunks x {seq_len}")
        return chunks

    config_name = "wikitext-103-raw-v1" if "103" in dataset_name else "wikitext-2-raw-v1"
    ds = load_dataset("wikitext", config_name, split=split)
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    print(f"  Tokenizing ({len(text):,} chars)...")
    tokens = tokenizer(text, return_tensors="pt").input_ids[0]
    if max_tokens and len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]

    n = len(tokens) // seq_len
    chunks = tokens[:n * seq_len].view(n, seq_len)
    print(f"  {n:,} chunks x {seq_len} = {n * seq_len:,} tokens")

    torch.save(chunks, cache_path)
    return chunks

print("--- Training Data ---")
train_chunks = load_and_chunk(tokenizer, "wikitext-103", "train", SEQ_LEN)

print("\n--- Eval Data ---")
eval_tokens_full = load_and_chunk(tokenizer, "wikitext-2", "test", EVAL_TOKENS)
# Flatten for PPL eval
eval_tokens = eval_tokens_full.reshape(-1)[:EVAL_TOKENS]

## 3. Phase 1: Unbounded Training

In [ ]:
# Evaluation functions

@torch.no_grad()
def eval_ppl(model, eval_tokens, device, max_length=2048, stride=512):
    """Compute perplexity with overlapping sliding window (unbounded)."""
    was_training = model.training
    model.eval()

    input_ids = eval_tokens.unsqueeze(0).to(device)
    seq_len = input_ids.size(1)
    nlls, n_tokens = [], 0

    for begin in range(0, seq_len - 1, stride):
        end = min(begin + max_length, seq_len)
        chunk = input_ids[:, begin:end]
        outputs = model(chunk, use_cache=False)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

        shift_logits = logits[:, :-1].contiguous()
        shift_labels = chunk[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1), reduction="none",
        )
        if begin > 0:
            skip = max_length - stride - 1
            if 0 < skip < loss.size(0):
                loss = loss[skip:]
        nlls.append(loss.sum())
        n_tokens += loss.numel()
        if end >= seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / n_tokens).item()
    if was_training:
        model.train()
    return ppl


@torch.no_grad()
def eval_ppl_sequential(model, adapters, eval_tokens, device,
                         chunk_size=2048, max_tokens=0):
    """Evaluate PPL with fresh state per chunk (for bounded models)."""
    was_training = model.training
    model.eval()

    tokens = eval_tokens[:max_tokens] if max_tokens > 0 else eval_tokens
    input_ids = tokens.unsqueeze(0).to(device)
    seq_len = input_ids.size(1)
    nlls, n_tokens = [], 0

    for start in range(0, seq_len, chunk_size):
        end = min(start + chunk_size, seq_len)
        if end - start < 2:
            break
        for a in adapters:
            a.reset_state()

        chunk = input_ids[:, start:end]
        outputs = model(chunk, use_cache=False)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

        shift_logits = logits[:, :-1].contiguous()
        shift_labels = chunk[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1), reduction="none",
        )
        nlls.append(loss.sum())
        n_tokens += loss.numel()

    ppl = torch.exp(torch.stack(nlls).sum() / n_tokens).item()
    for a in adapters:
        a.reset_state()
    if was_training:
        model.train()
    return ppl

print("Eval functions defined.")

In [ ]:
# Phase 1: Unbounded Training
print("=" * 60)
print(f"Phase 1: Unbounded Training ({PHASE1_STEPS} steps)")
print("=" * 60)

# Cosine schedule with linear warmup
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(WARMUP_STEPS, 1)
    progress = (step - WARMUP_STEPS) / max(PHASE1_STEPS - WARMUP_STEPS, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Shuffle training data
n_chunks = len(train_chunks)
perm = torch.randperm(n_chunks)
chunk_idx = 0

def get_batch():
    global chunk_idx, perm
    if chunk_idx + BATCH_SIZE > n_chunks:
        perm = torch.randperm(n_chunks)
        chunk_idx = 0
    batch = train_chunks[perm[chunk_idx:chunk_idx + BATCH_SIZE]].to(device)
    chunk_idx += BATCH_SIZE
    return batch

# Initial eval
ppl0 = eval_ppl(model, eval_tokens, device, max_length=SEQ_LEN)
print(f"  Step 0: eval PPL = {ppl0:.2f}")

# Training loop
model.train()
log = [{"step": 0, "eval_ppl": round(ppl0, 2)}]
best_ppl = ppl0
t0 = time.time()
accum_loss = 0.0
tokens_total = 0
print_every = max(GRAD_ACCUM * 5, GRAD_ACCUM)

optimizer.zero_grad()

for step in range(1, PHASE1_STEPS + 1):
    input_ids = get_batch()

    with torch.amp.autocast(device.type, dtype=dtype, enabled=(dtype != torch.float32)):
        outputs = model(input_ids, use_cache=False)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            input_ids[:, 1:].reshape(-1),
        )
        (loss / GRAD_ACCUM).backward()

    accum_loss += loss.item()
    tokens_total += input_ids.numel()

    if step % GRAD_ACCUM == 0:
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=MAX_GRAD_NORM,
        )
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        avg_loss = accum_loss / GRAD_ACCUM
        elapsed = time.time() - t0
        tok_s = tokens_total / elapsed if elapsed > 0 else 0
        lr_now = scheduler.get_last_lr()[0]
        gn = grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm

        entry = {
            "step": step,
            "loss": round(avg_loss, 4),
            "ppl_train": round(math.exp(min(avg_loss, 20)), 2),
            "grad_norm": round(gn, 4),
            "lr": lr_now,
        }
        log.append(entry)

        if step % print_every == 0:
            print(f"  Step {step:>6}: loss={avg_loss:.4f}  "
                  f"ppl={entry['ppl_train']:.1f}  "
                  f"gnorm={gn:.3f}  lr={lr_now:.2e}  "
                  f"{tok_s:.0f} tok/s")

        accum_loss = 0.0

    # Periodic eval
    if EVAL_EVERY and step % EVAL_EVERY == 0:
        ppl = eval_ppl(model, eval_tokens, device, max_length=SEQ_LEN)
        marker = "best" if ppl < best_ppl else f"+{ppl - best_ppl:.2f}"
        print(f"  Step {step}: eval PPL = {ppl:.2f} ({marker})")
        log.append({"step": step, "eval_ppl": round(ppl, 2)})
        if ppl < best_ppl:
            best_ppl = ppl
            # Save best checkpoint
            ckpt = {f"layer_{i}": a.state_dict() for i, a in enumerate(adapters)}
            torch.save(ckpt, OUTPUT_DIR / "phase1_best.pt")
        model.train()

# Final Phase 1 eval
final_ppl = eval_ppl(model, eval_tokens, device, max_length=SEQ_LEN)
print(f"\n  Final Phase 1 PPL: {final_ppl:.2f}")
log.append({"step": PHASE1_STEPS, "eval_ppl": round(final_ppl, 2), "phase": 1, "final": True})

# Save Phase 1 checkpoint
ckpt = {f"layer_{i}": a.state_dict() for i, a in enumerate(adapters)}
torch.save(ckpt, OUTPUT_DIR / "phase1_final.pt")

elapsed = time.time() - t0
print(f"Phase 1 complete: {elapsed/60:.1f} min, {tokens_total:,} tokens")
if device.type == "cuda":
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 4. Phase 2: Bounded Training

In [ ]:
# Phase 2: Switch to bounded attention and continue training
print("=" * 60)
print(f"Phase 2: Bounded Training ({PHASE2_STEPS} steps)")
print(f"  W={BOUNDED_WINDOW}, Me={BOUNDED_ME}, Ms={BOUNDED_MS}")
print("=" * 60)

# Switch unbounded adapters to bounded, copying trained weights
blocks = model.model.layers
adapters_b: List[Qwen3CoDAAdapter] = []

for i, (block, adapter_ub) in enumerate(zip(blocks, adapters)):
    coda_ub = adapter_ub.coda

    adapter_b = Qwen3CoDAAdapter(
        hidden_size=adapter_ub.hidden_size,
        num_heads=adapter_ub.num_heads,
        num_kv_heads=adapter_ub.num_kv_heads,
        head_dim=adapter_ub.head_dim,
        bounded=True,
        rope_theta=model.config.rope_theta,
        head_norm_mode=coda_ub.head_norm_mode,
        rope_interleaved=coda_ub.rope_interleaved,
        window=BOUNDED_WINDOW,
        num_landmarks_exact=BOUNDED_ME,
        num_landmarks_summary=BOUNDED_MS,
        block_size=BOUNDED_BLOCK_SIZE,
        detach_evicted=True,
    )
    adapter_b = adapter_b.to(device=device, dtype=dtype)
    coda_b = adapter_b.coda

    # Transfer trained weights
    with torch.no_grad():
        coda_b.q_proj.load_state_dict(coda_ub.q_proj.state_dict())
        coda_b.k_proj.load_state_dict(coda_ub.k_proj.state_dict())
        coda_b.v_proj.load_state_dict(coda_ub.v_proj.state_dict())
        coda_b.o_proj.load_state_dict(coda_ub.o_proj.state_dict())
        coda_b.lambda_proj.load_state_dict(coda_ub.lambda_proj.state_dict())
        coda_b.theta.copy_(coda_ub.theta)
        if (hasattr(coda_b, "head_norm") and hasattr(coda_ub, "head_norm")
                and not isinstance(coda_ub.head_norm, nn.Identity)):
            coda_b.head_norm.load_state_dict(coda_ub.head_norm.state_dict())

    block.self_attn = adapter_b
    adapters_b.append(adapter_b)

print(f"  Switched {len(adapters_b)} layers to bounded")

# Free old unbounded adapters
del adapters
if device.type == "cuda":
    torch.cuda.empty_cache()

# MUST disable gradient checkpointing for Phase 2
# (bounded state accumulation is incompatible with GC recomputation)
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
    print("  Gradient checkpointing: disabled for Phase 2")

In [ ]:
# Phase 2 optimizer (lower LR to avoid catastrophic forgetting)
for p in model.parameters():
    p.requires_grad = False

if FREEZE_MODE == "coda-only":
    for adapter in adapters_b:
        for name, p in adapter.named_parameters():
            if any(cn in name for cn in coda_param_names):
                p.requires_grad = True
elif FREEZE_MODE == "attention":
    for adapter in adapters_b:
        for p in adapter.parameters():
            p.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {n_train:,}")

lr_p2 = LR * PHASE2_LR_SCALE
lr_coda_p2 = LR_CODA * PHASE2_LR_SCALE

adapter_param_ids = set()
for adapter in adapters_b:
    for p in adapter.parameters():
        adapter_param_ids.add(id(p))

coda_p2, proj_p2, other_p2 = [], [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if id(p) in adapter_param_ids:
        if any(cn in name for cn in coda_param_names):
            coda_p2.append(p)
        else:
            proj_p2.append(p)
    else:
        other_p2.append(p)

groups_p2 = []
if coda_p2:
    groups_p2.append({"params": coda_p2, "lr": lr_coda_p2, "weight_decay": 0.0})
    print(f"  CoDA params: {sum(p.numel() for p in coda_p2):,} @ lr={lr_coda_p2}")
if proj_p2:
    groups_p2.append({"params": proj_p2, "lr": lr_p2, "weight_decay": WEIGHT_DECAY})
    print(f"  Proj params: {sum(p.numel() for p in proj_p2):,} @ lr={lr_p2}")
if other_p2:
    groups_p2.append({"params": other_p2, "lr": lr_p2, "weight_decay": WEIGHT_DECAY})

optimizer_p2 = torch.optim.AdamW(groups_p2)

In [ ]:
# Phase 2 training loop

def lr_lambda_p2(step):
    warmup = min(50, PHASE2_STEPS // 10)
    if step < warmup:
        return step / max(warmup, 1)
    progress = (step - warmup) / max(PHASE2_STEPS - warmup, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler_p2 = torch.optim.lr_scheduler.LambdaLR(optimizer_p2, lr_lambda_p2)

# Reset data iterator
perm = torch.randperm(n_chunks)
chunk_idx = 0

# Phase 2 baseline eval
ppl_p2_start = eval_ppl_sequential(
    model, adapters_b, eval_tokens, device,
    chunk_size=SEQ_LEN, max_tokens=EVAL_BOUNDED_TOKENS,
)
print(f"  Phase 2 start bounded PPL: {ppl_p2_start:.2f}")
log.append({"step": PHASE1_STEPS, "phase": 2, "bounded_ppl_start": round(ppl_p2_start, 2)})

# Training
model.train()
t0_p2 = time.time()
accum_loss = 0.0
tokens_total_p2 = 0
best_ppl_p2 = ppl_p2_start

optimizer_p2.zero_grad()

for step in range(1, PHASE2_STEPS + 1):
    input_ids = get_batch()

    # Reset state before each forward
    for a in adapters_b:
        a.reset_state()

    with torch.amp.autocast(device.type, dtype=dtype, enabled=(dtype != torch.float32)):
        outputs = model(input_ids, use_cache=False)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            input_ids[:, 1:].reshape(-1),
        )
        (loss / GRAD_ACCUM).backward()

    accum_loss += loss.item()
    tokens_total_p2 += input_ids.numel()

    if step % GRAD_ACCUM == 0:
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=MAX_GRAD_NORM,
        )
        optimizer_p2.step()
        scheduler_p2.step()
        optimizer_p2.zero_grad()

        avg_loss = accum_loss / GRAD_ACCUM
        elapsed = time.time() - t0_p2
        tok_s = tokens_total_p2 / elapsed if elapsed > 0 else 0
        gn = grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm

        if step % print_every == 0:
            print(f"  P2 Step {step:>6}: loss={avg_loss:.4f}  "
                  f"ppl={math.exp(min(avg_loss, 20)):.1f}  "
                  f"gnorm={gn:.3f}  {tok_s:.0f} tok/s")

        accum_loss = 0.0

    if EVAL_EVERY and step % EVAL_EVERY == 0:
        ppl = eval_ppl_sequential(
            model, adapters_b, eval_tokens, device,
            chunk_size=SEQ_LEN, max_tokens=EVAL_BOUNDED_TOKENS,
        )
        marker = "best" if ppl < best_ppl_p2 else f"+{ppl - best_ppl_p2:.2f}"
        print(f"  P2 Step {step}: bounded PPL = {ppl:.2f} ({marker})")
        log.append({"step": PHASE1_STEPS + step, "phase": 2, "eval_ppl": round(ppl, 2)})
        if ppl < best_ppl_p2:
            best_ppl_p2 = ppl
            ckpt = {f"layer_{i}": a.state_dict() for i, a in enumerate(adapters_b)}
            torch.save(ckpt, OUTPUT_DIR / "phase2_best.pt")
        model.train()

# Final Phase 2 eval
final_ppl_p2 = eval_ppl_sequential(
    model, adapters_b, eval_tokens, device,
    chunk_size=SEQ_LEN, max_tokens=EVAL_BOUNDED_TOKENS,
)
print(f"\n  Final Phase 2 bounded PPL: {final_ppl_p2:.2f}")
log.append({"step": PHASE1_STEPS + PHASE2_STEPS, "phase": 2, "bounded_ppl": round(final_ppl_p2, 2), "final": True})

# Save Phase 2 checkpoint
ckpt = {f"layer_{i}": a.state_dict() for i, a in enumerate(adapters_b)}
torch.save(ckpt, OUTPUT_DIR / "phase2_final.pt")

elapsed_p2 = time.time() - t0_p2
print(f"Phase 2 complete: {elapsed_p2/60:.1f} min, {tokens_total_p2:,} tokens")
if device.type == "cuda":
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 5. Results Summary

In [ ]:
# Summary
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

ppls = [e["eval_ppl"] for e in log if "eval_ppl" in e]
losses = [e["loss"] for e in log if "loss" in e]

if losses:
    print(f"Train loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
if ppls:
    print(f"Eval PPL:   {ppls[0]:.2f} -> {ppls[-1]:.2f}")

bounded_entries = [e for e in log if "bounded_ppl" in e]
if bounded_entries:
    print(f"Bounded PPL: {bounded_entries[-1]['bounded_ppl']:.2f}")

# Save log
(OUTPUT_DIR / "training_log.json").write_text(
    json.dumps(log, indent=2, default=str), encoding="utf-8",
)
print(f"\nLog saved to {OUTPUT_DIR / 'training_log.json'}")

## 6. Push to Hugging Face Hub

Publishes the full model (base weights + CoDA adapters) as a standalone model.

In [ ]:
# Login to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Save the full model for publishing
# We save the complete model state (base + CoDA adapters) so it can be
# loaded directly without needing the original Qwen3-4B weights.

SAVE_DIR = OUTPUT_DIR / "hf_model"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving model to {SAVE_DIR}...")

# Save adapter weights separately (for users who want to apply to their own base)
adapter_state = {f"layer_{i}": a.state_dict() for i, a in enumerate(adapters_b)}
torch.save(adapter_state, SAVE_DIR / "coda_adapters.pt")
print(f"  Adapter weights: {SAVE_DIR / 'coda_adapters.pt'}")

# Save tokenizer
tokenizer.save_pretrained(SAVE_DIR)
print(f"  Tokenizer saved")

# Save CoDA config
coda_config = {
    "base_model": MODEL_NAME,
    "attention_type": "CoDA-GQA-L",
    "num_layers_swapped": len(adapters_b),
    "bounded": True,
    "window": BOUNDED_WINDOW,
    "num_landmarks_exact": BOUNDED_ME,
    "num_landmarks_summary": BOUNDED_MS,
    "block_size": BOUNDED_BLOCK_SIZE,
    "rope_theta": model.config.rope_theta,
    "rope_interleaved": False,
    "head_norm_mode": "identity",
    "head_dim": adapters_b[0].head_dim,
    "num_heads": adapters_b[0].num_heads,
    "num_kv_heads": adapters_b[0].num_kv_heads,
    "hidden_size": adapters_b[0].hidden_size,
    "training": {
        "phase1_steps": PHASE1_STEPS,
        "phase2_steps": PHASE2_STEPS,
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "lr_coda": LR_CODA,
        "freeze_mode": FREEZE_MODE,
        "dataset": "wikitext-103",
    },
}

ppls_final = [e["eval_ppl"] for e in log if "eval_ppl" in e]
bounded_final = [e for e in log if "bounded_ppl" in e]
if ppls_final:
    coda_config["eval_ppl_unbounded"] = ppls_final[-1] if not bounded_final else ppls_final[0]
if bounded_final:
    coda_config["eval_ppl_bounded"] = bounded_final[-1]["bounded_ppl"]

(SAVE_DIR / "coda_config.json").write_text(
    json.dumps(coda_config, indent=2, default=str), encoding="utf-8",
)
print(f"  CoDA config saved")
print(f"\nReady to push to {HF_REPO_ID}")

In [ ]:
# Create model card

model_card = f"""---
license: apache-2.0
base_model: {MODEL_NAME}
tags:
  - coda-gqa-l
  - differential-attention
  - bounded-kv-cache
  - qwen3
---

# Qwen3-4B + CoDA-GQA-L

Qwen3-4B with **CoDA-GQA-L** (Constrained Orthogonal Differential Attention +
Grouped-Query Attention with Landmark Memory) attention replacement.

## What is CoDA-GQA-L?

CoDA-GQA-L replaces standard attention with two innovations:

1. **Differential Attention**: Computes signal and inhibitory streams from the
   same query via orthogonal rotation, subtracting noise weighted by a learned
   gate. Sharpens attention without a second Wq projection.

2. **Bounded KV Memory**: Replaces O(L) KV cache with O(W + Me + Ms) per layer:
   - **Recent window** (W={BOUNDED_WINDOW}): Ring buffer of exact recent tokens
   - **Exact landmark bank** (Me={BOUNDED_ME}): Novelty-filtered LRU cache
   - **Summary landmark bank** (Ms={BOUNDED_MS}): EMA prototypes compressing older context

## Training

- **Base model**: `{MODEL_NAME}`
- **Phase 1**: {PHASE1_STEPS} steps unbounded (teaches differential attention)
- **Phase 2**: {PHASE2_STEPS} steps bounded (teaches memory banks)
- **Dataset**: WikiText-103
- **Sequence length**: {SEQ_LEN}
- **Freeze mode**: {FREEZE_MODE} (only attention params trainable)

## Usage

```python
from coda_gqa_l import Qwen3CoDAAdapter

# Load adapter weights onto Qwen3-4B
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{MODEL_NAME}", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("{MODEL_NAME}")

# Swap attention layers
adapters = Qwen3CoDAAdapter.swap_qwen3_layers(model, bounded=True)

# Load trained weights
state = torch.load("coda_adapters.pt", map_location="cpu")
for i, adapter in enumerate(adapters):
    adapter.load_state_dict(state[f"layer_{{i}}"], strict=False)
```

## Architecture

| Parameter | Value |
|-----------|-------|
| Base model | Qwen3-4B |
| Hidden size | {adapters_b[0].hidden_size} |
| Q heads | {adapters_b[0].num_heads} |
| KV heads | {adapters_b[0].num_kv_heads} |
| Head dim | {adapters_b[0].head_dim} |
| Layers | {len(adapters_b)} |
| Window (W) | {BOUNDED_WINDOW} |
| Exact bank (Me) | {BOUNDED_ME} |
| Summary bank (Ms) | {BOUNDED_MS} |
| rope_theta | {model.config.rope_theta:,} |
"""

(SAVE_DIR / "README.md").write_text(model_card, encoding="utf-8")
print("Model card written.")

In [ ]:
# Push to Hugging Face Hub
from huggingface_hub import HfApi

api = HfApi()

# Create the repo if it doesn't exist
api.create_repo(HF_REPO_ID, exist_ok=True)

# Upload all files
api.upload_folder(
    folder_path=str(SAVE_DIR),
    repo_id=HF_REPO_ID,
    commit_message=f"Qwen3-4B + CoDA-GQA-L (Phase 1: {PHASE1_STEPS} steps, Phase 2: {PHASE2_STEPS} steps)",
)

print(f"\nPublished to: https://huggingface.co/{HF_REPO_ID}")

## 7. Quick Inference Test

In [ ]:
# Quick generation test with bounded attention
model.eval()
for a in adapters_b:
    a.reset_state()

prompt = "The key advantage of bounded KV caches is that"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

print(f"Prompt: {prompt}")
print(f"Generating with bounded CoDA (W={BOUNDED_WINDOW}, Me={BOUNDED_ME}, Ms={BOUNDED_MS})...")
print()

with torch.no_grad():
    generated = model.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

output = tokenizer.decode(generated[0], skip_special_tokens=True)
print(output)